# Research Notebook Template: Dual SMA WFA (BTC-USD)

Este notebook funciona como **template de iteracion**.

- Las funciones y logica principal viven en `example_bt_optimo.py`.
- Aqui solo hacemos configuracion, ejecucion, tablas de analisis e interpretacion.
- Objetivo: menos codigo repetido y mas lectura de resultados OOS.

In [10]:
# Si falta alguna dependencia, descomenta la linea siguiente:
# !pip install -q vectorbt yfinance plotly

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from example_bt_optimo import (
    fetch_close_prices,
    run_walk_forward,
    build_benchmark,
    compute_alpha_decay,
    as_scalar,
    SYMBOL,
    START,
    END,
    INITIAL_CASH,
    FEE,
    WF_WINDOW_BARS,
    IS_RATIO,
    FAST_WINDOWS,
    SLOW_WINDOWS,
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

In [11]:
# Parametros de iteracion (puedes sobrescribir los defaults importados)
CFG = {
    'symbol': SYMBOL,
    'start': START,
    'end': END,
    'initial_cash': INITIAL_CASH,
    'fee': FEE,
    'wf_window_bars': WF_WINDOW_BARS,
    'is_ratio': IS_RATIO,
    'fast_windows': FAST_WINDOWS,
    'slow_windows': SLOW_WINDOWS,
}

pd.DataFrame([CFG])

,symbol,start,end,initial_cash,fee,wf_window_bars,is_ratio,fast_windows,slow_windows
0,BTC-USD,2017-01-01,None,100000,0.001,500,0.8,"(5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60...","(20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120..."


In [12]:
# Descarga de datos y grafica de precio
close = fetch_close_prices(SYMBOL, START, END)
display(close.tail())

fig_price = px.line(close, x=close.index, y=close.values, title=f'{SYMBOL} Close Price (Daily)')
fig_price.update_layout(template='plotly_white', xaxis_title='Date', yaxis_title='Price (USD)')
fig_price.show()

Date
2026-03-01    65738.101562
2026-03-02    68775.851562
2026-03-03    68293.648438
2026-03-04    72710.578125
2026-03-06    71244.000000
Name: BTC-USD, dtype: float64

In [13]:
# Ejecucion Walk-Forward unanchored
close = fetch_close_prices(CFG['symbol'], CFG['start'], CFG['end'])
strategy_pf, wf_results, close_oos_all = run_walk_forward(
    close=close,
    wf_window_bars=CFG['wf_window_bars'],
    is_ratio=CFG['is_ratio'],
    fast_windows=CFG['fast_windows'],
    slow_windows=CFG['slow_windows'],
    init_cash=CFG['initial_cash'],
    fees=CFG['fee'],
)

benchmark_pf = build_benchmark(close_oos_all, init_cash=CFG['initial_cash'], fees=CFG['fee'])
alpha_decay_pct = compute_alpha_decay(wf_results)

resumen_wf = pd.DataFrame({
    'Metric': [
        'Total ventanas OOS',
        'Sharpe IS promedio',
        'Sharpe OOS promedio',
        'Alpha Decay (%)',
    ],
    'Valor': [
        len(wf_results),
        wf_results['is_sharpe'].replace([np.inf, -np.inf], np.nan).mean(),
        wf_results['oos_sharpe'].replace([np.inf, -np.inf], np.nan).mean(),
        alpha_decay_pct,
    ],
})

display(resumen_wf.style.format({'Valor': '{:,.4f}'}))
display(wf_results.head(10))

print('Interpretacion rapida:')
print('- Si Alpha Decay es alto y positivo, hay deterioro de la ventaja al pasar de IS a OOS.')
print('- Si OOS Sharpe promedio es bajo o negativo, conviene revisar robustez de parametros y costos.')

,Metric,Valor
0,Total ventanas OOS,29.0000
1,Sharpe IS promedio,2.2376
2,Sharpe OOS promedio,-0.0464
3,Alpha Decay (%),102.0727


,window_id,is_start,is_end,oos_start,oos_end,best_fast,best_slow,is_sharpe,oos_sharpe
0,1,2017-01-01,2018-02-04,2018-02-05,2018-05-15,10,20,3.958488,0.385258
1,2,2017-04-11,2018-05-15,2018-05-16,2018-08-23,10,20,3.069932,-0.880645
2,3,2017-07-20,2018-08-23,2018-08-24,2018-12-01,10,20,1.871478,-2.419807
3,4,2017-10-28,2018-12-01,2018-12-02,2019-03-11,5,190,inf,inf
4,5,2018-02-05,2019-03-11,2019-03-12,2019-06-19,5,180,inf,inf
5,6,2018-05-16,2019-06-19,2019-06-20,2019-09-27,35,90,2.719601,inf
6,7,2018-08-24,2019-09-27,2019-09-28,2020-01-05,40,50,2.237842,-1.882968
7,8,2018-12-02,2020-01-05,2020-01-06,2020-04-14,30,60,2.054055,-2.524254
8,9,2019-03-12,2020-04-14,2020-04-15,2020-07-23,5,20,1.593847,-0.365239
9,10,2019-06-20,2020-07-23,2020-07-24,2020-10-31,10,70,0.715375,3.601405


Interpretacion rapida:
- Si Alpha Decay es alto y positivo, hay deterioro de la ventaja al pasar de IS a OOS.
- Si OOS Sharpe promedio es bajo o negativo, conviene revisar robustez de parametros y costos.


In [14]:
# KPIs y tablas comparativas
strategy_sharpe = as_scalar(strategy_pf.sharpe_ratio())
strategy_mdd = as_scalar(strategy_pf.max_drawdown())
strategy_ret = as_scalar(strategy_pf.total_return())

benchmark_sharpe = as_scalar(benchmark_pf.sharpe_ratio())
benchmark_mdd = as_scalar(benchmark_pf.max_drawdown())
benchmark_ret = as_scalar(benchmark_pf.total_return())

kpi_table = pd.DataFrame({
    'Metric': ['Total Return', 'Sharpe Ratio', 'Max Drawdown'],
    'Strategy OOS': [strategy_ret, strategy_sharpe, strategy_mdd],
    'Benchmark B&H': [benchmark_ret, benchmark_sharpe, benchmark_mdd],
})

display(
    kpi_table.style.format({
        'Strategy OOS': '{:,.4f}',
        'Benchmark B&H': '{:,.4f}',
    })
)

display(strategy_pf.stats().to_frame('Strategy').head(12))
display(benchmark_pf.stats().to_frame('Benchmark').head(12))

print('Comentario interpretativo:')
print('- El benchmark captura la tendencia secular de BTC; la estrategia debe evaluarse por eficiencia de riesgo (Sharpe/MDD), no solo retorno bruto.')

,Metric,Strategy OOS,Benchmark B&H
0,Total Return,-0.0926,12.6776
1,Sharpe Ratio,0.1668,0.8402
2,Max Drawdown,-0.7035,-0.7663


,Strategy
Start,2018-02-05 00:00:00
End,2026-01-13 00:00:00
Period,2900 days 00:00:00
Start Value,100000.0
End Value,90741.432965
Total Return [%],-9.258567
Benchmark Return [%],1270.497205
Max Gross Exposure [%],100.0
Total Fees Paid,5729.484605
Max Drawdown [%],70.346339


,Benchmark
Start,2018-02-05 00:00:00
End,2026-01-13 00:00:00
Period,2900 days 00:00:00
Start Value,100000.0
End Value,1367758.949307
Total Return [%],1267.758949
Benchmark Return [%],1270.497205
Max Gross Exposure [%],100.0
Total Fees Paid,1469.028177
Max Drawdown [%],76.634564


Comentario interpretativo:
- El benchmark captura la tendencia secular de BTC; la estrategia debe evaluarse por eficiencia de riesgo (Sharpe/MDD), no solo retorno bruto.


In [15]:
# Curva de equity: estrategia vs benchmark
strategy_equity = strategy_pf.value()
benchmark_equity = benchmark_pf.value()

fig_eq = go.Figure()
fig_eq.add_trace(go.Scatter(x=strategy_equity.index, y=strategy_equity.values, mode='lines', name='Dual SMA WFA (OOS Concat)', line=dict(width=2)))
fig_eq.add_trace(go.Scatter(x=benchmark_equity.index, y=benchmark_equity.values, mode='lines', name='Benchmark Buy & Hold', line=dict(width=2, dash='dot')))
fig_eq.update_layout(
    title='Walk-Forward Unanchored: Strategy vs Benchmark',
    xaxis_title='Date',
    yaxis_title='Equity (USD)',
    template='plotly_white',
    hovermode='x unified'
)
fig_eq.show()

In [17]:
# Tabla de robustez por ventana (prioridad research)
robustez = wf_results.copy()
robustez['delta_sharpe'] = robustez['oos_sharpe'] - robustez['is_sharpe']
robustez['regimen'] = np.where(robustez['oos_sharpe'] > 0, 'OOS positivo', 'OOS debil')

display(
    robustez[['window_id', 'best_fast', 'best_slow', 'is_sharpe', 'oos_sharpe', 'delta_sharpe', 'regimen']]
    .sort_values('oos_sharpe', ascending=False)
    .head(12)
)

print('Comentario interpretativo:')
print('- Ventanas con delta_sharpe muy negativo sugieren sobreajuste local en IS.')
print('- Repeticion de (best_fast, best_slow) en ventanas estables sugiere mayor robustez estructural.')

,window_id,best_fast,best_slow,is_sharpe,oos_sharpe,delta_sharpe,regimen
4,5,5,180,inf,inf,NaN,OOS positivo
3,4,5,190,inf,inf,NaN,OOS positivo
5,6,35,90,2.719601,inf,inf,OOS positivo
15,16,20,80,0.865332,inf,inf,OOS positivo
16,17,5,240,inf,inf,NaN,OOS positivo
17,18,5,160,inf,inf,NaN,OOS positivo
18,19,5,130,inf,inf,NaN,OOS positivo
20,21,50,60,1.437398,4.096976,2.659578,OOS positivo
9,10,10,70,0.715375,3.601405,2.886030,OOS positivo
10,11,15,30,2.268352,3.164946,0.896594,OOS positivo


Comentario interpretativo:
- Ventanas con delta_sharpe muy negativo sugieren sobreajuste local en IS.
- Repeticion de (best_fast, best_slow) en ventanas estables sugiere mayor robustez estructural.


In [18]:
# Visualizacion compacta de estabilidad de parametros
freq_fast = wf_results['best_fast'].value_counts().rename_axis('best_fast').reset_index(name='count')
freq_slow = wf_results['best_slow'].value_counts().rename_axis('best_slow').reset_index(name='count')

display(freq_fast.head(10))
display(freq_slow.head(10))

fig_oos = px.box(
    wf_results,
    y='oos_sharpe',
    points='all',
    title='Distribucion de OOS Sharpe por Ventana'
    )
fig_oos.update_layout(template='plotly_white', yaxis_title='OOS Sharpe')
fig_oos.show()

print('Comentario interpretativo:')
print('- Una distribucion OOS muy dispersa implica sensibilidad al regimen de mercado.')
print('- Parametros dominantes en frecuencia pueden usarse como punto de partida para un set parsimonioso.')

,best_fast,count
0,5,7
1,10,5
2,40,3
3,35,2
4,15,2
5,25,2
6,20,2
7,65,2
8,50,2
9,30,1


,best_slow,count
0,60,6
1,20,5
2,40,4
3,90,3
4,70,2
5,80,2
6,50,1
7,190,1
8,180,1
9,30,1


Comentario interpretativo:
- Una distribucion OOS muy dispersa implica sensibilidad al regimen de mercado.
- Parametros dominantes en frecuencia pueden usarse como punto de partida para un set parsimonioso.


In [20]:
# Convertir en función y agregar en example_bt_optimo.py para facilitar reuso en otros activos o marcos temporales.
# Visualizacion research 1: decay IS vs OOS por ventana
plot_df = wf_results.copy()
plot_df['decay'] = plot_df['oos_sharpe'] - plot_df['is_sharpe']

fig_decay = go.Figure()
fig_decay.add_trace(go.Scatter(x=plot_df['window_id'], y=plot_df['is_sharpe'], mode='lines+markers', name='IS Sharpe'))
fig_decay.add_trace(go.Scatter(x=plot_df['window_id'], y=plot_df['oos_sharpe'], mode='lines+markers', name='OOS Sharpe'))
fig_decay.update_layout(
    title='Sharpe por Ventana: IS vs OOS',
    xaxis_title='Window ID',
    yaxis_title='Sharpe Ratio',
    template='plotly_white'
)
fig_decay.show()

fig_bar = px.bar(plot_df, x='window_id', y='decay', title='Sharpe Delta por Ventana (OOS - IS)')
fig_bar.update_layout(template='plotly_white', xaxis_title='Window ID', yaxis_title='Sharpe Delta')
fig_bar.show()